# Imports

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
from scipy import ndimage
from skimage import filters, measure, morphology

# Slike so v mapi vaje 09 (deluje iz 10_vaje/ in iz 10_vaje/resitve/)
GRADIVO = Path("../09_vaje/navodila/Gradivo")
if not GRADIVO.exists():
    GRADIVO = Path("../../09_vaje/navodila/Gradivo")

# Funkcije

In [ ]:
def zgladi(img, sigma):
    """Gaussovo glajenje; vrne float sliko v intervalu [0, 1]."""
    return filters.gaussian(img, sigma=sigma) if sigma > 0 else img / 255.0


def prag(img_f, razredi=2):
    """Otsujev prag (razredi = 2) ali spodnji prag Otsujeve metode s tremi razredi (razredi = 3)."""
    # TODO: filters.threshold_otsu oz. filters.threshold_multiotsu(..., classes=3)[0]
    pass


def maska_otsu(img_f, razredi=2):
    """Temni objekti: piksli pod pragom."""
    return img_f < prag(img_f, razredi)


def element_disk(r):
    """Strukturni element: disk s polmerom r (bool array (2r+1) x (2r+1))."""
    # TODO: x^2 + y^2 <= r^2
    pass


def dilatacija(M, E):
    """Binarna dilatacija: piksel je 1, če element, postavljen vanj, prekrije vsaj en piksel maske."""
    r = E.shape[0] // 2
    H, W = M.shape
    P = np.pad(M, r)
    out = np.zeros_like(M)
    # TODO: za vsak premik (di, dj), kjer je E == 1, ALI z ustrezno premaknjeno masko
    pass
    return out


def erozija(M, E):
    """Binarna erozija: piksel ostane 1, če element, postavljen vanj, v celoti leži v maski."""
    r = E.shape[0] // 2
    H, W = M.shape
    P = np.pad(M, r, constant_values=True)   # zunaj slike privzamemo ospredje
    out = np.ones_like(M)
    # TODO: IN namesto ALI
    pass
    return out


def odpiranje(M, E):
    # TODO: erozija, nato dilatacija
    pass


def zapiranje(M, E):
    # TODO: dilatacija, nato erozija
    pass


def segmentiraj(img, sigma=5, r=3, A_min=100, razredi=3):
    """Celoten postopek: glajenje, prag, zapolnitev lukenj, odpiranje, označevanje, odstranitev majhnih.
    Vrne (oznake, seznam regij)."""
    # TODO
    pass


def okroglost(reg):
    """Okroglost 4 pi A / P^2 (1 za krog)."""
    return 4 * np.pi * reg.area / reg.perimeter**2

# Testno območje

In [ ]:
img = cv2.imread(str(GRADIVO / "Kontrola.tif"), cv2.IMREAD_UNCHANGED)
print(img.shape, img.dtype, img.min(), img.max())

# P2: erozija in dilatacija majhnega kvadrata s križem 3 x 3
M = np.zeros((7, 7), bool)
M[2:5, 2:5] = True
kriz = element_disk(1)
print(kriz.astype(int))
print("dilatacija:", dilatacija(M, kriz).sum(), " erozija:", erozija(M, kriz).sum())
print("skimage:   ", morphology.binary_dilation(M, kriz).sum(), morphology.binary_erosion(M, kriz).sum())

# P4: okroglost kroga in kvadrata
print("krog r=20:", okroglost(measure.regionprops(element_disk(20).astype(int))[0]))

# Naloga 1 - Slika 2

In [ ]:
fig = plt.figure(figsize=(16, 4))
ax = fig.add_subplot(1, 5, 1)
ax.hist(img.ravel(), bins=256, range=(0, 256), density=True)
ax.set_xlabel("sivina")
# TODO: pragova (2 in 3 razredi) za sigma = 0, 2, 5, 10; število območij; maske s 3 razredi (Preglednica 1)
plt.show()

# Naloga 2 - Slika 3

In [ ]:
izrez = maska_otsu(zgladi(img, 2))[150:300, 300:450]
E = element_disk(3)
operacije = {"maska": izrez, "erozija": erozija(izrez, E), "dilatacija": dilatacija(izrez, E),
             "odpiranje": odpiranje(izrez, E), "zapiranje": zapiranje(izrez, E)}
fig = plt.figure(figsize=(16, 3.5))
# TODO: prikaz vseh operacij; število pikslov in območij; primerjava s skimage (Preglednica 2)
plt.show()

# Naloga 3 - Slika 4

In [ ]:
L, regije = segmentiraj(img, sigma=5, r=3, A_min=100)
L2, regije2 = segmentiraj(img, sigma=5, r=3, A_min=100, razredi=2)
print("dva razreda:", len(regije2), "objektov, okroglost < 0.5:", sum(okroglost(r) < 0.5 for r in regije2))
print("število objektov:", len(regije))
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111)
ax.imshow(img, cmap="gray")
# TODO: obrisi objektov, težišča in očrtani pravokotniki
ax.set_title(f"{len(regije)} objektov")
ax.axis("off")
plt.show()

In [ ]:
# Preglednica 3: trije največji objekti
# TODO

# Naloga 4 - Slika 5

In [ ]:
d = np.array([r.equivalent_diameter_area for r in regije])
c = np.array([okroglost(r) for r in regije])
fig = plt.figure(figsize=(11, 4))
# TODO: histograma ekvivalentnega premera in okroglosti; povprečja in std (Preglednica 4)
fig.tight_layout()
plt.show()

In [ ]:
# Preglednica 4: občutljivost na parametra sigma in A_min
# TODO